# Midstone 2 — Prepare the Data (QuickPlate)

**Author:** Jay K Jacob

### 1. Import the required modules

In [1]:
import numpy as np
import pandas as pd
import operator

### 2. Unzip `midstone-02-your-data.zip`

In [2]:
import os, zipfile

ZIP = "midstone-02-your-data.zip"

if os.path.exists("orders"):
    print("Already unzipped - skipping.")
else:
    with zipfile.ZipFile(ZIP) as z:
        z.extractall()
    print("Unzipped.")

Already unzipped - skipping.


### 3. Read all 7 CSV files

In [3]:
customers = pd.read_csv("customers/customers.csv")
menu_items = pd.read_csv("menu_items/menu_items.csv")
order_items_01 = pd.read_csv("order_items/day=01/order_items_01.csv")
payments_01 = pd.read_csv("payments/day=01/payments_01.csv")
restaurants = pd.read_csv("restaurants/restaurants.csv")
riders = pd.read_csv("riders/riders.csv")
orders_01 = pd.read_csv("orders/day=01/orders_01.csv")

### 4. Q1 · Number of rows in each file

In [4]:
print(f"Shape(row, col) of DB customers: {customers.shape}")

Shape(row, col) of DB customers: (20000, 4)


In [5]:
print(f"Shape(row, col) of DB menu_items: {menu_items.shape}")

Shape(row, col) of DB menu_items: (8000, 4)


In [6]:
print(f"Shape(row, col) of DB orders_01: {orders_01.shape}")

Shape(row, col) of DB orders_01: (5112, 10)


In [7]:
print(f"Shape(row, col) of DB payments_01: {payments_01.shape}")

Shape(row, col) of DB payments_01: (5367, 6)


In [8]:
print(f"Shape(row, col) of DB restaurants: {restaurants.shape}")

Shape(row, col) of DB restaurants: (500, 6)


In [9]:
print(f"Shape(row, col) of DB riders: {riders.shape}")

Shape(row, col) of DB riders: (300, 4)


In [10]:
print(f"Shape(row, col) of DB order_items_01: {order_items_01.shape}")

Shape(row, col) of DB order_items_01: (13946, 5)


### 5. Problems in `orders_01.csv` (Q2–Q6)

##### Check the columns, types and empty values in `orders_01`

In [11]:
orders_01.info()

<class 'pandas.DataFrame'>
RangeIndex: 5112 entries, 0 to 5111
Data columns (total 10 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   order_id       5112 non-null   str    
 1   customer_id    5095 non-null   str    
 2   restaurant_id  5101 non-null   str    
 3   rider_id       4653 non-null   str    
 4   order_ts       5112 non-null   str    
 5   promised_ts    5112 non-null   str    
 6   delivered_ts   4653 non-null   str    
 7   status         5112 non-null   str    
 8   total_amount   5112 non-null   float64
 9   updated_at     5112 non-null   str    
dtypes: float64(1), str(9)
memory usage: 399.5 KB


#### Q2 · Order IDs that repeat
How many order_id values appear more than once

*Method: count each `order_id` with `groupby`, then count the IDs that appear more than once.*

In [12]:
not_uniq = (orders_01.groupby("order_id")["order_id"].count() > 1).sum()
print(f"No.of Order IDs that repeat: {not_uniq}")

No.of Order IDs that repeat: 42


#### Q3 · Unknown restaurant
restaurant_id is empty, or isn't in restaurants.csv

*Method 1: empty IDs + IDs not found in `restaurants.csv` (`~isin`) dropping NA values.*


*Method 2: IDs not found in `restaurants.csv` (`~isin`) without dropping NA values.*

In [13]:
rest_id_empty = orders_01["restaurant_id"].isna().sum()
rest_id_not_present = (~orders_01["restaurant_id"].dropna().isin(restaurants["restaurant_id"])).sum()
rest_id_unknown = rest_id_empty + rest_id_not_present
print(f"No.of Unknown restaurant IDs: {rest_id_unknown}")

No.of Unknown restaurant IDs: 19


In [14]:
rest_id_not_present = (~orders_01["restaurant_id"].isin(restaurants["restaurant_id"])).sum()
print(f"No.of Unknown restaurant IDs: {rest_id_not_present}")

No.of Unknown restaurant IDs: 19


#### Q4 · Total below 0
total_amount is less than 0

In [15]:
tot_less_than_0 = (orders_01["total_amount"] < 0).sum()
print(f"No.of total_amount values less than 0: {tot_less_than_0}")

No.of total_amount values less than 0: 8


#### Q5 · Time written the other way
order_ts written like 01/03/2026 14:07

*Method: rows that contain `/` are in the other format.*

Look at the first 10 values to see the time formats.

In [16]:
orders_01["order_ts"].head(10)

0    2026-03-01 00:01:00
1    2026-03-01 00:02:00
2    2026-03-01 00:06:00
3    2026-03-01 00:09:00
4       01/03/2026 00:11
5    2026-03-01 00:12:00
6    2026-03-01 00:14:00
7    2026-03-01 00:16:00
8    2026-03-01 00:16:00
9       01/03/2026 00:18
Name: order_ts, dtype: str

In [17]:
orders_01["order_ts"].isna().sum()

np.int64(0)

In [18]:
orders_01["order_ts"].count()

np.int64(5112)

In [19]:
day_format = 0
for ts in orders_01["order_ts"]:
 if operator.contains(ts, "/"):
     day_format += 1
print(f"No. order_ts written like 01/03/2026 14:07 : {day_format}")

No. order_ts written like 01/03/2026 14:07 : 776


#### Q6 · No customer
customer_id is empty

In [20]:
no_customer = orders_01["customer_id"].isnull().sum()
print(f"No.of customer IDs empty: {no_customer}")

No.of customer IDs empty: 17
